# NB03_fPINN — Closed loop on the fractional plant: CTC + I with every identified model

**Purpose.** Third step of the fractional-PINN paper: closed-loop validation of the identified models. The controller is the baseline CTC + I of the earlier notebooks (same torque-domain gains `Kp = 150`, `Kd = 10`, `Ki/Kp = 10` for every model, 2 kHz), so differences in tracking come only from the inverse-dynamics model. It is run on the Adept path at the three speeds and 0 / 6 kg, on the **fractional plant** of NB01_fPINN / NB02_fPINN

`τ = γ·Y·θ0 + s·τ_u + B·D^α q`   (`s = 1`, `α ∈ {0.3, 0.5, 0.7, 0.9}`, `B = 0.05 N m s^α`),

with the networks trained in NB02_fPINN (MLP, HYBRID, PINN, fPINN, fHYBRID; 35 / 175 trajectories; seeds) plus the nominal model and a fractional **oracle** (exact τ_u and exact B·D^α q, lower bound).

**How the fractional term enters the controller.** The plant keeps the full history of q at 2 kHz and evaluates `D^α q` with the L1 scheme (full memory). A fractional model splits into a memoryless part `F(q, q̇, q̈, m)`, used as the CTC inverse dynamics exactly as in the baseline CTC, and the memory term `B̂·D^α̂ q`, evaluated **online from the measured history** at 2 ms and added to the torque. Integer models have `B̂ = 0`, so for them the controller is exactly the baseline CTC + I.

**One switch.** `QUICK = True` is a fast check (~2 min: Adept at scale 3.2, 0 kg, the `_e3` networks of a QUICK run of NB02_fPINN). `QUICK = False` runs **the whole closed-loop set of the paper**, reading the `_e150` weights that NB02_fPINN (`QUICK = False`) leaves in `WEIGHTS_DIR`, with nothing else to edit:

| Block | α | Models | References | Controllers | Runs |
|---|---|---|---|---|---|
| Main study | 0.5 | nominal, oracle, 5 networks × 35/175 traj × 3 seeds | Adept × 3 speeds × 0/6 kg | CTC + I | 192 |
| α sweep | 0.3, 0.7, 0.9 | nominal, oracle, 4 networks (175 traj, seed 0) | same | same | 108 |

**Expected runtime (Ryzen 7 PRO 5850U, 16 GB):** about 15–30 min (QUICK here: ~2 s per run at the middle speed; the slow reference is twice as long). Results are appended to `NB03_fPINN_ClosedLoop.csv` after every run and the notebook skips runs already in it, so an interruption loses at most one run. Missing weight files are skipped with a message. `TAU_MAX = 5 N m` per joint (continuous motor rating 0.64 N m × 10:1 gearbox × 0.9 efficiency); the fastest reference with 6 kg saturates the actuators, so that case is reported but should be read with care.

**Outputs:** `NB03_fPINN_ClosedLoop.csv` (one row per run: α, model, data size, seed, speed, payload, Cartesian RMSE / max error, RMS torque, TV, peak torque), `NB03_fPINN_Summary.csv`, `fig_nb03_fpinn_rmse_vs_speed.png`.

In [ ]:
import os, time, numpy as np, pandas as pd
from math import gamma as Gamma
import matplotlib
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK        = True                # True: fast check (~2 min). False: the whole closed-loop set of the paper (see header).
SEVERITY     = 1.0
B_FRAC       = 0.05                # [N m s^α] per joint
SPEED_SCALES = (6.4, 3.2, 1.6)     # time scaling of the Adept path (16, 8, 4 s)
PAYLOADS     = (0.0, 6.0)          # kg; known to every model-based controller
KINDS        = ("MLP", "HYBRID", "PINN", "fPINN", "fHYBRID")
ALPHA_MAIN   = 0.5                 # same blocks as NB02_fPINN
MAIN_SIZES, MAIN_SEEDS = (35, 175), (0, 1, 2)
ALPHA_SWEEP  = (0.3, 0.7, 0.9)
SWEEP_KINDS, SWEEP_SIZES, SWEEP_SEEDS = ("PINN", "HYBRID", "fPINN", "fHYBRID"), (175,), (0,)
EPOCHS_TAG   = 150                 # epoch budget in the NB02_fPINN file names
WEIGHTS_DIR  = "."                 # folder with the NB02_fPINN_*.npz files
FS  = 2000; dt = 1.0 / FS          # plant integration (Euler-Cromer), as in nb 18
T_C = 0.002                        # period at which the memory term is updated [s]
TAU_MAX = 5.0                      # [N m] per joint (continuous rating 0.64 N m x 10:1 x 0.9 = 5.8 N m)
if QUICK:                          # uses the _e3 weights of a QUICK run of NB02_fPINN
    SPEED_SCALES, PAYLOADS, MAIN_SIZES, MAIN_SEEDS, EPOCHS_TAG = (3.2,), (0.0,), (35,), (0,), 3
    ALPHA_SWEEP, SWEEP_KINDS, SWEEP_SIZES, SWEEP_SEEDS = (0.3,), ("fPINN",), (35,), (0,)
ALPHAS_ALL = (ALPHA_MAIN,) + tuple(ALPHA_SWEEP)
def nets_for(alpha):
    if alpha == ALPHA_MAIN: return [(k, n, sd) for k in KINDS for n in MAIN_SIZES for sd in MAIN_SEEDS]
    return [(k, n, sd) for k in SWEEP_KINDS for n in SWEEP_SIZES for sd in SWEEP_SEEDS]
# -----------------------------------------------------------------------------------------------
_n = sum(2 + len(nets_for(a)) for a in ALPHAS_ALL) * len(SPEED_SCALES) * len(PAYLOADS)
print("QUICK" if QUICK else "FULL", f"| alphas {ALPHAS_ALL} | scales {SPEED_SCALES} | payloads {PAYLOADS} | {_n} closed-loop runs")

## Robot model, unmodeled dynamics (notebook 18) and the fractional plant

`FracHistory` evaluates `D^α q(t_n) = h^-α/Γ(2-α) Σ_k b_k Δq_{n-k}` incrementally (full memory, one dot product per step). `plant_qdd` is that of NB01 with the term `B·D^α q` added.

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)

def gamma_payload(m): return (M_BASE + float(m)) / M_BASE

def fk(q):
    rd = R_BASE - R_PLAT; C = np.zeros((3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[i])
        C[i] = [r*np.cos(a), r*np.sin(a), L_A*np.sin(q[i])]
    A1, B1, C1 = 2*(C[1]-C[0]); D1 = 2*L_A*rd*(np.cos(q[0])-np.cos(q[1]))
    A2, B2, C2 = 2*(C[2]-C[0]); D2 = 2*L_A*rd*(np.cos(q[0])-np.cos(q[2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2-A2*D1)/den; L2 = (A1*C2-A2*C1)/den; L3 = (B2*D1-B1*D2)/den; L4 = (B2*C1-B1*C2)/den
    u = 1 + L2**2 + L4**2
    v = 2*(L3*L4 + L1*L2 - C[0,0]*L4 - C[0,1]*L2 - C[0,2])
    w = C[0,0]**2 + C[0,1]**2 + C[0,2]**2 + L3**2 + L1**2 - 2*C[0,0]*L3 - 2*C[0,1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.array([L3 + L4*z, L1 + L2*z, z])

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def regressor(q, a, p):
    x, y, z = p; s, c = np.sin(q), np.cos(q); Y = np.zeros((3, 14))
    Y[:, 0] = a; Y[:, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[i, cx] = -x*s[i]; Y[i, cy] = -y*s[i]; Y[i, cz] = z*c[i]; Y[i, cs] = s[i]
    return Y

def tau_unmodeled(q, qd, sev):
    u = UNMODELED
    return sev*(u["b_v"]*qd + u["f_c"]*np.tanh(qd/u["eps"])
                + u["c_cor"]*(qd**2*np.cos(q) + qd[[1, 2, 0]]*qd[[2, 0, 1]])
                + u["k_pos"]*np.sin(u["w_pos"]*q))

def plant_qdd(q, qd, tau, p, m, sev):
    g = gamma_payload(m)
    return (tau - g*(regressor(q, np.zeros(3), p) @ THETA0) - tau_unmodeled(q, qd, sev)) / (g*THETA0[0])

# Inverse-dynamics models F(q, qd, qdd, m) (p = FK(q) inside F)
def F_nominal(q, qd, qdd, m):
    return gamma_payload(m) * (regressor(q, qdd, fk(q)) @ THETA0)

def make_F_oracle(sev):
    return lambda q, qd, qdd, m: F_nominal(q, qd, qdd, m) + tau_unmodeled(q, qd, sev)

class FracHistory:
    """Online Caputo derivative (L1 scheme, full memory) of a 3-vector sampled every h seconds from t = 0."""
    def __init__(self, alpha, h, n_max):
        self.alpha, self.h = alpha, h
        k = np.arange(n_max + 1); self.b = ((k + 1.0)**(1 - alpha) - k**(1.0 - alpha))[::-1].copy()   # reversed weights
        self.c = h**(-alpha) / Gamma(2 - alpha); self.dq = np.zeros((n_max + 1, 3)); self.n = 0; self.q_last = None
    def push(self, q):
        """Append the sample q(t_n) and return D^α q(t_n)."""
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        self.n += 1; self.dq[self.n] = q - self.q_last; self.q_last = q.copy()
        return self.c * (self.b[-self.n:] @ self.dq[1:self.n + 1])

def caputo_l1(x, h, alpha):
    """Batch version (reference signals): x (N, 3) from t = 0."""
    from scipy.signal import fftconvolve
    N = len(x); k = np.arange(N); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, 3)), np.diff(x, axis=0)])
    return fftconvolve(d, b[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

def plant_qdd_frac(q, qd, tau, p, m, sev, Dq):
    """NB01 plant + fractional damper; Dq = D^α q(t) from the plant history."""
    g = gamma_payload(m)
    return (tau - g*(regressor(q, np.zeros(3), p) @ THETA0) - tau_unmodeled(q, qd, sev) - B_FRAC*Dq) / (g*THETA0[0])

# check of the online derivative against the batch one
_q = np.column_stack([np.sin(np.linspace(0, 3, 801))]*3); _H = FracHistory(0.5, dt, 800)
_on = np.array([_H.push(v) for v in _q]); print("online vs batch D^α: max diff %.1e" % np.abs(_on - caputo_l1(_q, dt, 0.5)).max())

## References: Adept path (NB01)

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05

def adept_reference(scale):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]
    P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(t=t, P=P, q=q, qd=qd, qdd=qdd, N=n)

REFS = {sc: adept_reference(sc) for sc in SPEED_SCALES}
for sc, R in REFS.items():
    print(f"scale {sc}: {R['t'][-1]:.1f} s | peak |qd| = {np.abs(R['qd']).max():.2f} rad/s | peak |qdd| = {np.abs(R['qdd']).max():.1f} rad/s^2")

## Models: nominal, fractional oracle, and the NB02_fPINN networks

Every model is a pair: memoryless inverse dynamics `F(q, q̇, q̈, m)` (as in NB02) and memory gains `(α̂, B̂)`; `B̂ = 0` for nominal / MLP / HYBRID / PINN. The oracle has the exact `τ_u` and the exact `(α, B)`.

In [ ]:
def silu(z): return z / (1.0 + np.exp(-z))

class NumpyFF:
    """Forward pass of notebook 18 / NB02 (kind MLP, HYBRID, PINN; fHYBRID, fPINN use the same core)."""
    def __init__(self, path):
        d = np.load(path, allow_pickle=False); self.kind = str(d["kind"]); self.core = self.kind.lstrip("f") if self.kind.startswith("f") else self.kind
        self.W = [d[k].astype(float) for k in ("W1", "b1", "W2", "b2", "W3", "b3")]
        self.x_mean, self.x_std = d["x_mean"].astype(float), d["x_std"].astype(float)
        self.y_mean, self.y_std, self.r_std = d["y_mean"].astype(float), d["y_std"].astype(float), d["r_std"].astype(float)
        self.alpha, self.B = float(d["alpha"]), d["B"].astype(float)
    def net(self, xs):
        W1, b1, W2, b2, W3, b3 = self.W
        return W3 @ silu(W2 @ silu(W1 @ xs + b1) + b2) + b3
    def ff(self, q, qd, qdd, m, p):
        out = self.net((np.concatenate([q, qd, qdd, [m], p]) - self.x_mean) / self.x_std)
        if self.core == "MLP":    return out*self.y_std + self.y_mean
        if self.core == "HYBRID": return gamma_payload(m)*(regressor(q, qdd, p) @ THETA0) + out*self.r_std
        return regressor(q, qdd, p) @ out
    def as_F(self): return lambda q, qd, qdd, m: self.ff(q, qd, qdd, m, fk(q))

def load_models(alpha):
    M = [dict(label="nominal", F=F_nominal, alpha=0.5, B=np.zeros(3), size=0, seed=0),
         dict(label="oracle (frac)", F=make_F_oracle(SEVERITY), alpha=alpha, B=np.full(3, B_FRAC), size=0, seed=0)]
    for kind, size, seed in nets_for(alpha):
        f = os.path.join(WEIGHTS_DIR, f"NB02_fPINN_{kind}_{size}traj_seed{seed}_a{alpha}_e{EPOCHS_TAG}.npz")
        if not os.path.exists(f): print("missing:", f); continue
        n = NumpyFF(f); M.append(dict(label=kind, F=n.as_F(), alpha=n.alpha if n.B.any() else 0.5, B=n.B, size=size, seed=seed))
        if n.B.any(): print(f"alpha {alpha}: {kind} {size} traj seed {seed}: alpha_hat = {n.alpha:.3f}, B_hat = {np.round(n.B, 4)}")
    return M
MODELS = {a: load_models(a) for a in ALPHAS_ALL}
print({a: len(M) for a, M in MODELS.items()}, "models per alpha")

## Controller with the memory term

`CTCIF` is the baseline CTC + I. `WithMemory` adds `B̂·D^α̂ q` evaluated from the measured joint angles at the control rate (2 ms); with `B̂ = 0` it is a pass-through.

In [ ]:
KP_PD, KD_PD, KI_RATIO = 150.0, 10.0, 10.0          # same torque-domain gains as the baseline CTC
KP_EQ, KD_EQ = KP_PD/THETA0[0], KD_PD/THETA0[0]
class CTCIF:
    def __init__(self, F, name): self.F, self.name = F, name
    def prepare(self, R, m): pass
    def reset(self): self.I = np.zeros(3)
    def __call__(self, R, i, q, qd, p, m):
        e = R["q"][i] - q; ed = R["qd"][i] - qd; self.I += e*dt
        v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*self.I
        return self.F(q, qd, v, m)

class WithMemory:
    def __init__(self, inner, alpha, B):
        self.inner, self.alpha, self.B, self.name = inner, alpha, np.asarray(B, float), inner.name
        self.step = int(round(T_C/dt))
    def prepare(self, R, m):
        self.inner.prepare(R, m); self.n_max = R["N"] // self.step + 2
        for a in ("L", "t_prepare", "solve_times", "active", "n_solves"):
            if hasattr(self.inner, a): setattr(self, a, getattr(self.inner, a))
    def reset(self):
        self.inner.reset(); self.H = FracHistory(self.alpha, T_C, self.n_max) if self.B.any() else None; self.mem = np.zeros(3)
        if hasattr(self.inner, "solve_times"): self.solve_times = self.inner.solve_times
    def __call__(self, R, i, q, qd, p, m):
        if self.H is not None and i % self.step == 0: self.mem = self.B*self.H.push(q)
        tau = self.inner(R, i, q, qd, p, m) + self.mem
        if hasattr(self.inner, "active"): self.active, self.n_solves = self.inner.active, self.inner.n_solves
        return tau

## Closed-loop simulation on the fractional plant (metrics of NB01)

In [ ]:
def simulate(ctrl, R, m, sev, alpha):
    ctrl.reset(); q = R["q"][0].copy(); qd = R["qd"][0].copy(); Hp = FracHistory(alpha, dt, R["N"])
    P = np.zeros((R["N"], 3)); T = np.zeros((R["N"], 3))
    for i in range(R["N"]):
        p = fk(q); P[i] = p; Dq = Hp.push(q)
        tau = np.clip(ctrl(R, i, q, qd, p, m), -TAU_MAX, TAU_MAX); T[i] = tau
        qd = qd + plant_qdd_frac(q, qd, tau, p, m, sev, Dq)*dt; q = q + qd*dt
        if not np.all(np.isfinite(q)) or np.abs(q).max() > 3: return None
    e = 1000*np.linalg.norm(R["P"] - P, axis=1)
    out = {"Cartesian RMSE (mm)": np.sqrt(np.mean(e**2)), "Max error (mm)": e.max(),
           "RMS torque (N m)": np.sqrt(np.mean(T**2)), "TV (N m)": np.sum(np.linalg.norm(np.diff(T, axis=0), axis=1)),
           "Peak |tau| (N m)": np.abs(T).max()}
    if hasattr(ctrl, "solve_times") and len(ctrl.solve_times):
        st = 1e3*np.array(ctrl.solve_times)
        out.update({"Online step mean (ms)": st.mean(), "Online step p95 (ms)": np.percentile(st, 95),
                    "Active-constraint cycles (%)": 100*ctrl.active/max(ctrl.n_solves, 1),
                    "min eig F_qdd": ctrl.L["minlam"], "Offline prepare (s)": ctrl.t_prepare})
    return out, e, T

## Run: every model × reference × controller

In [ ]:
rows = []; traces = {}; t_all = time.time(); OUT_CSV = "NB03_fPINN_ClosedLoop" + ("_QUICK" if QUICK else "") + ".csv"   # QUICK writes apart, so its rows never block the full run
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()          # resume after an interruption
key = lambda d: (d["alpha true"], d["Controller"], d["Model"], int(d["Data (traj.)"]), int(d["Seed"]), float(d["Speed scale"]), float(d["Payload (kg)"]))
done_keys = {key(r) for _, r in done.iterrows()} if len(done) else set(); rows = done.to_dict("records")
scen = [(sc, m) for sc in SPEED_SCALES for m in PAYLOADS]; k_run = 0
for alpha in ALPHAS_ALL:
    for sc, m in scen:
        R = REFS[sc]
        for M in MODELS[alpha]:
            for cname, inner in (("CTC + I", CTCIF(M["F"], "CTC + I")),):
                k_run += 1
                base = {"alpha true": alpha, "Controller": cname, "Model": M["label"], "Data (traj.)": M["size"], "Seed": M["seed"],
                        "Speed scale": sc, "Payload (kg)": m, "alpha hat": M["alpha"] if M["B"].any() else np.nan, "B hat mean": M["B"].mean()}
                if key(base) in done_keys: continue
                c = WithMemory(inner, M["alpha"], M["B"]); t0 = time.time(); c.prepare(R, m); res = simulate(c, R, m, SEVERITY, alpha)
                base["Run time (s)"] = time.time() - t0
                if res is None: rows.append(dict(base, Unstable=True)); print(f"[{k_run}/{_n}] {cname} {M['label']} UNSTABLE")
                else:
                    met, e, T = res; traces[(alpha, cname, M["label"], M["size"], M["seed"], sc, m)] = e
                    rows.append(dict(base, Unstable=False, **met))
                    print(f"[{k_run}/{_n}] α {alpha} | scale {sc} | {m:.0f} kg | {cname} | {M['label']:13s} {M['size']:3d} traj s{M['seed']} | "
                          f"RMSE {met['Cartesian RMSE (mm)']:.4f} mm | {time.time()-t0:.0f} s | {(time.time()-t_all)/60:.1f} min", flush=True)
                pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
df = pd.DataFrame(rows)
print(f"total {(time.time()-t_all)/60:.1f} min | unstable runs: {int(df['Unstable'].sum())}")

## Summary: Cartesian RMSE and improvement over the nominal model

In [ ]:
pd.set_option("display.width", 250)
df["Label"] = df["Model"] + np.where(df["Data (traj.)"] > 0, " [" + df["Data (traj.)"].astype(str) + " traj]", "")
ok = df[~df["Unstable"]]
summ = ok.groupby(["alpha true", "Controller", "Label", "Payload (kg)", "Speed scale"])["Cartesian RMSE (mm)"].agg(["mean", "std"]).reset_index()
summ.to_csv("NB03_fPINN_Summary.csv", index=False)
for alpha in ALPHAS_ALL:
    for cname in ("CTC + I",):
        pv = summ[(summ["alpha true"] == alpha) & (summ.Controller == cname)].pivot(index="Label", columns=["Payload (kg)", "Speed scale"], values="mean")
        if pv.empty: continue
        print(f"\nα = {alpha} | {cname}: Cartesian RMSE (mm), mean over seeds"); print(pv.round(4).to_string())
        if "nominal" in pv.index: print("improvement factor vs nominal (>1 = better):"); print((pv.loc["nominal"] / pv).round(2).to_string())

## Figure: RMSE vs peak joint speed (CTC + I, 175 trajectories)

In [ ]:
speed = {sc: float(np.abs(REFS[sc]["qd"]).max()) for sc in SPEED_SCALES}
size_fig = max(MAIN_SIZES)
show = [("nominal", "o", "0.4"), (f"MLP [{size_fig} traj]", "s", "tab:blue"), (f"HYBRID [{size_fig} traj]", "^", "tab:green"),
        (f"PINN [{size_fig} traj]", "D", "tab:orange"), (f"fHYBRID [{size_fig} traj]", "v", "tab:olive"),
        (f"fPINN [{size_fig} traj]", "P", "tab:red"), ("oracle (frac)", "x", "k")]
fig, axes = plt.subplots(1, len(PAYLOADS), figsize=(7.0, 2.6), sharey=True, squeeze=False)
for ax, m in zip(axes[0], PAYLOADS):
    for lab, mk, col in show:
        sub = summ[(summ["alpha true"] == ALPHA_MAIN) & (summ.Controller == "CTC + I") & (summ.Label == lab) & (summ["Payload (kg)"] == m)].sort_values("Speed scale", ascending=False)
        if sub.empty: continue
        ax.errorbar([speed[s] for s in sub["Speed scale"]], sub["mean"], yerr=sub["std"].fillna(0), marker=mk, ms=4, capsize=2, lw=1, color=col, label=lab)
    ax.set_yscale("log"); ax.set_xlabel("Peak joint speed (rad/s)"); ax.set_title(f"CTC + I, α = {ALPHA_MAIN}, payload {m:.0f} kg", fontsize=9); ax.grid(True, which="both", alpha=0.3)
axes[0][0].set_ylabel("Cartesian RMSE (mm)"); axes[0][0].legend(fontsize=6, loc="upper left")
fig.tight_layout(); fig.savefig("fig_nb03_fpinn_rmse_vs_speed.png", dpi=200); fig.savefig("fig_nb03_fpinn_rmse_vs_speed.pdf"); plt.show()